# Lesson 10: Supervised Sentiment Analysis on Twitter

In this lesson, we build a complete supervised classification workflow for text sentiment analysis. We will work with a gold-standard dataset of tweets and train classical machine learning algorithms to classify them into Positive, Negative, or Neutral sentiments.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand Supervised Learning**: Understand how classifiers learn patterns from human-annotated gold standard data.
2. **Handle Real-World Data Cleaning**: Deal with parsing anomalies, handle custom delimiters, and drop missing values (`NaN`) that cause vectorizer crashes.
3. **Represent Text as Numerical Features**: Implement the **Bag-of-Words** approach using `CountVectorizer` and understand its structural limits.
4. **Train and Evaluate Classifiers**: Train a Bayesian model (**Naive Bayes**) and an ensemble model (**Random Forest**).
5. **Evaluate Model Correctness**: Understand evaluation baselines, calculate accuracy scores, and perform detailed error diagnostics.
6. **Refactor Code into Classes**: Abstract machine learning workflows into a clean, reusable Python class.

---
## 1. Downloading the Dataset

We will use the [**Twitter Entity Sentiment Analysis**](https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis) dataset from Kaggle.

We download the ZIP archive containing the CSV files directly using `curl`.

In [1]:
### Address of the database: https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis
!curl -L -o twitter-entity-sentiment-analysis.zip \
  https://www.kaggle.com/api/v1/datasets/download/jp797498e/twitter-entity-sentiment-analysis

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


 13 2041k   13  266k    0     0   189k      0  0:00:10  0:00:01  0:00:09  189k

100 2041k  100 2041k    0     0  1188k      0  0:00:01  0:00:01 --:--:-- 5725k


---
## 2. Opening and Parsing CSV Files in Zip Archives

Rather than decompressing the zip archive onto the disk, we can read the files directly from memory using the built-in `zipfile.ZipFile` module.

### CSV Quotation Pitfalls
Tweets contain punctuation, including commas. In comma-separated value (CSV) files, a comma is normally interpreted as a separator. If a tweet text contains a comma, the CSV reader will incorrectly split it into multiple columns, causing alignment errors.

To prevent this, CSV specifications allow enclosing fields in a **quotation character** (usually double quotes `"`). We instruct Pandas to handle this by passing `quotechar='"'`.

In [2]:
from zipfile import ZipFile
import pandas as pd
import numpy as np

# Open the zip archive and list contained files
zip_path = 'twitter-entity-sentiment-analysis.zip'
arch = ZipFile(zip_path)
print("Files inside the archive:", arch.namelist())

Files inside the archive: ['twitter_training.csv', 'twitter_validation.csv']


### Loading Datasets and Renaming Columns
We load both the training set (`twitter_training.csv`) and validation set (`twitter_validation.csv`), and assign columns: `ID`, `ENTITY`, `SENTIMENT`, and `TEXT`.

In [3]:
df_train = pd.read_csv(arch.open('twitter_training.csv'), header=None, quotechar='"')
df_val = pd.read_csv(arch.open('twitter_validation.csv'), header=None, quotechar='"')

cols = ["ID", "ENTITY", "SENTIMENT", "TEXT"]
df_train.columns = cols
df_val.columns = cols

print(f"Training set shape  : {df_train.shape}")
print(f"Validation set shape: {df_val.shape}")


Training set shape  : (74682, 4)
Validation set shape: (1000, 4)


---
## 3. Data Quality & Preprocessing

Real-world datasets contain formatting issues and noise that must be cleaned before training machine learning models.

### 3.1 Handling Missing Values (`NaN`)
If a tweet was empty or missing in the dataset, Pandas imports it as a `NaN` floating-point value. Scikit-learn's vectorizers expect strings. Passing a float/NaN to `CountVectorizer` causes a `ValueError` crash. We clean this by dropping missing texts using `.dropna(subset=["TEXT"]).

### 3.2 Filtering Out 'Irrelevant' Sentiments
Some data points are labeled as `Irrelevant` (representing topics unrelated to the query entity). We remove them to keep our classification task focused on a 3-way sentiment: `Positive`, `Negative`, and `Neutral`.

In [4]:
# Inspect missing values
print("Rows with null text in training set:")
print(df_train[df_train["TEXT"].isnull()])

# Clean missing values in-place
df_train.dropna(subset=["TEXT"], inplace=True)
df_val.dropna(subset=["TEXT"], inplace=True)

# Filter out the 'Irrelevant' labels
df_train = df_train[df_train["SENTIMENT"] != "Irrelevant"]
df_val = df_val[df_val["SENTIMENT"] != "Irrelevant"]

print(f"\nAfter cleaning:")
print(f"  Cleaned training shape  : {df_train.shape}")
print(f"  Cleaned validation shape: {df_val.shape}")


Rows with null text in training set:
         ID       ENTITY SENTIMENT TEXT
61     2411  Borderlands   Neutral  NaN
553    2496  Borderlands   Neutral  NaN
589    2503  Borderlands   Neutral  NaN
745    2532  Borderlands  Positive  NaN
1105   2595  Borderlands  Positive  NaN
...     ...          ...       ...  ...
73972  9073       Nvidia  Positive  NaN
73973  9073       Nvidia  Positive  NaN
74421  9154       Nvidia  Positive  NaN
74422  9154       Nvidia  Positive  NaN
74423  9154       Nvidia  Positive  NaN

[686 rows x 4 columns]

After cleaning:
  Cleaned training shape  : (61121, 4)
  Cleaned validation shape: (828, 4)


### 3.3 Sentiment Class Distribution
We inspect the balance of categories in our training set using the `value_counts()` method.

In [5]:
print(df_train["SENTIMENT"].value_counts())

SENTIMENT
Negative    22358
Positive    20655
Neutral     18108
Name: count, dtype: int64


### 3.4 Extracting Parallel Lists
We extract text sequences and labels into separate lists. We must ensure the symmetry is perfectly preserved: the item at index `i` in the texts list must match the label at index `i` in the labels list.

In [6]:
txt_train = df_train["TEXT"].tolist()
lbl_train = df_train["SENTIMENT"].tolist()

txt_val = df_val["TEXT"].tolist()
lbl_val = df_val["SENTIMENT"].tolist()

print(f"Parallel training lists size  : {len(txt_train)}")
print(f"Parallel validation lists size: {len(txt_val)}")


Parallel training lists size  : 61121
Parallel validation lists size: 828


---
## 4. Text Representation: Bag-of-Words

To train machine learning models, text must be translated into numerical feature vectors. In the **Bag-of-Words (BoW)** model, a document is represented as a vector of word counts.

### Document-Term Matrix
For a corpus with vocabulary size $V$, every document is represented as a $V$-dimensional vector containing count frequencies. The entire corpus forms a sparse matrix of size $N \times V$.

### Limitations
Since word counts are aggregated, word order is entirely discarded. For instance, **"the dog bit the owner"** and **"the owner bit the dog"** end up with identical vector representations, despite having opposite meanings.

We instantiate `CountVectorizer` from `scikit-learn` to fit the vocabulary and extract features.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

# Fit vectorizer on training text and transform it into a sparse matrix
vects = CountVectorizer()
X = vects.fit_transform(txt_train)

print(f"Document-Term Matrix shape: {X.shape}")
print(f"Vocabulary size: {len(vects.vocabulary_):,} unique words")

Document-Term Matrix shape: (61121, 26837)
Vocabulary size: 26,837 unique words


---
## 5. Bayesian Classification with Naive Bayes

The **Naive Bayes** classifier uses Bayes' theorem to compute the probability of a class given the words:
$$P(c \mid d) \propto P(c) \prod_{i} P(w_i \mid c)$$
It is called "naive" because it assumes that the occurrence of each word is independent of other words, given the class. Despite this simplification, it works exceptionally well for text classification.

We use `MultinomialNB` from `scikit-learn` to train our classifier.

In [8]:
from sklearn.naive_bayes import MultinomialNB

# Fit the classifier
clf = MultinomialNB()
clf.fit(X, lbl_train)
print("Naive Bayes classifier trained successfully.")

Naive Bayes classifier trained successfully.


### 5.1 Testing Custom Predictions
Let's test our classifier on some toy sentences. Note that the inputs must be transformed using the already-fitted vectorizer before being sent to the classifier's `predict()` method.

In [9]:
test_texts = [
    "Johnny is a horrible player",
    "Anna is a marvelous player"
]
X_test = vects.transform(test_texts)
predictions = clf.predict(X_test)

for text, pred in zip(test_texts, predictions):
    print(f"  '{text}' -> Predicted: {pred}")

  'Johnny is a horrible player' -> Predicted: Negative
  'Anna is a marvelous player' -> Predicted: Neutral


### 5.2 Evaluating Accuracy on the Validation Set
We predict the labels for the validation set and evaluate the accuracy. The random baseline for a 3-class classification is **33.3%**.

In [10]:
from sklearn.metrics import accuracy_score

# Transform the validation texts
X_val = vects.transform(txt_val)

# Predict labels
y_pred = clf.predict(X_val)

# Calculate accuracy
acc = accuracy_score(lbl_val, y_pred)
print(f"Naive Bayes Validation Accuracy: {acc:.4%}")

Naive Bayes Validation Accuracy: 85.5072%


---
## 6. Ensemble Methods: Random Forest

**Random Forest** is an ensemble learning method that builds multiple decision trees on random subsets of features and data points (bagging), taking a majority vote for the final prediction.
It is highly expressive and robust to overfitting, though it requires more computation than Naive Bayes.

In [11]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest
clf_rf = RandomForestClassifier()
clf_rf.fit(X, lbl_train)
print("Random Forest classifier trained successfully.")

Random Forest classifier trained successfully.


In [12]:
# Evaluate Random Forest
y_pred_rf = clf_rf.predict(X_val)
acc_rf = accuracy_score(lbl_val, y_pred_rf)
print(f"Random Forest Validation Accuracy: {acc_rf:.4%}")

Random Forest Validation Accuracy: 97.4638%


---
## 7. Refactoring the Pipeline into a Python Class

Instead of writing redundant preprocessing, fitting, and scoring code across multiple cells, we can encapsulate our logic in a reusable Python class. This reduces the surface area for bugs and improves reproducibility.

We have packaged this into `sentiment_classifier.py`. Let's import the class and execute the entire supervised pipeline in a few lines.

In [13]:
from sentiment_classifier import SentimentClassifier

zip_path = 'twitter-entity-sentiment-analysis.zip'

# 1. Load data directly from zip
df_train_raw = SentimentClassifier.load_raw_csv_from_zip(zip_path, 'twitter_training.csv')
df_val_raw = SentimentClassifier.load_raw_csv_from_zip(zip_path, 'twitter_validation.csv')

# 2. Instantiate and clean
pipeline = SentimentClassifier(vectorizer_type="count", classifier_type="naive_bayes")
train_texts, train_labels = pipeline.preprocess_df(df_train_raw)
val_texts, val_labels = pipeline.preprocess_df(df_val_raw)

# 3. Fit and evaluate
pipeline.fit(train_texts, train_labels)
score = pipeline.evaluate(val_texts, val_labels)
print(f"Polished Pipeline Naive Bayes Accuracy: {score:.4%}")

Polished Pipeline Naive Bayes Accuracy: 85.5072%


---
## Exercises

Complete the following exercises to deepen your understanding of sentiment classification. Focus on writing clean, readable Python code and document your findings in the markdown slots provided.

### Exercise 1 — Negation & structural limits in Bag-of-Words

We discussed how word order is lost in Bag-of-Words representations.

1. Predict the sentiment of these two sentences using your trained Naive Bayes classifier:
   - Sentence A: *"The movie was not good, it was bad."*
   - Sentence B: *"The movie was not bad, it was good."*
2. Print the predicted categories.
3. Explain why the model behaves this way despite the sentences having opposite sentiments.

In [14]:
# Your code here
sentences = [
    "The movie was not good, it was bad.",
    "The movie was not bad, it was good."
]
# TODO: Use the fitted vectorizer and classifier to make predictions


*Your observations:*

- 

### Exercise 2 — TF-IDF features

By default, `CountVectorizer` counts raw word frequencies. Common function words (e.g. "the", "and") can dominate count vectors. **TF-IDF** (Term Frequency-Inverse Document Frequency) down-weights words that are common across the entire corpus and highlights words that carry distinctive information.

1. Instantiate a new `SentimentClassifier` using `vectorizer_type="tfidf"` and `classifier_type="naive_bayes"`.
2. Train it and evaluate the validation accuracy.
3. Compare it with the count vectorizer version. Does TF-IDF improve performance? Explain why or why not.

In [15]:
# Your code here
# TODO: Train TF-IDF Naive Bayes pipeline and evaluate validation accuracy


*Your observations:*

- 

### Exercise 3 — Classification Report & Detailed Metrics

Validation accuracy provides an overall correctness score, but does not tell us where the model makes errors.

1. Generate a detailed metrics report using `sklearn.metrics.classification_report` on the validation set for the Naive Bayes model.
2. Discuss the difference in performance (Precision, Recall, F1) between `Positive`, `Negative`, and `Neutral` classes. Which class is the most difficult to classify correctly?

In [16]:
from sklearn.metrics import classification_report

# Your code here
# TODO: Generate and print classification report comparing lbl_val and NB predictions


*Your observations:*

- 

### Exercise 4 — Hyperparameter Tuning

We can pass configuration settings to both our text vectorizers and classifiers.

1. Experiment with **three different parameter configurations** using `SentimentClassifier`.
   *Hint: you can pass `vectorizer_params={'max_df': 0.95, 'ngram_range': (1, 2)}` or `classifier_params={'alpha': 0.1}` to the classifier constructor.*
2. Print the validation accuracy for each trial.
3. Explain the theoretical effect of adding bi-grams (`ngram_range=(1,2)`) or tuning Naive Bayes smoothing (`alpha`) on model performance.

In [17]:
# Your code here
# TODO: Instantiate and evaluate 3 variations of your pipeline


*Your observations:*

- 